# Train XLM-RoBERTa on UIT-VSFC

This notebook clones the merged `main` branch, installs the training environment, fine-tunes the model with early stopping, evaluates it, and creates downloadable artifacts. Kaggle Internet and a GPU must be enabled.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

workspace = Path('/kaggle/working')
repo = workspace / 'AI_in_DevOps-DataOps-MLOps_Final_Project'
repo_url = 'https://github.com/nhienthai/AI_in_DevOps-DataOps-MLOps_Final_Project.git'

if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', repo_url, str(repo)], check=True)

os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-training.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Repository ready:', repo)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle Notebook settings before training.')

print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([
    sys.executable, 'scripts/train_model.py',
    '--model-type', 'transformer',
    '--model-name', 'xlm-roberta-base',
    '--dataset', 'tridm/UIT-VSFC',
    '--epochs', '10',
    '--batch-size', '16',
    '--lr', '2e-5',
    '--output-dir', './artifacts/xlm-roberta',
    '--mlflow-uri', 'sqlite:///mlflow.db',
], check=True)

In [ ]:
model_path = repo / 'artifacts' / 'xlm-roberta' / 'xlm-roberta'
subprocess.run([
    sys.executable, 'scripts/evaluate_model.py',
    '--model-path', str(model_path),
    '--split', 'test',
    '--show-wrong', '20',
    '--output-csv', './artifacts/eval_results.csv',
], check=True)

In [ ]:
import shutil
import zipfile

model_archive = shutil.make_archive(
    str(workspace / 'model_weights'),
    'zip',
    root_dir=model_path.parent,
    base_dir=model_path.name,
)
mlflow_archive = workspace / 'mlflow_results.zip'
with zipfile.ZipFile(mlflow_archive, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(repo / 'mlflow.db', arcname='mlflow.db')
    archive.write(repo / 'artifacts' / 'eval_results.csv', arcname='eval_results.csv')

print('Created:', model_archive, mlflow_archive)

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(workspace / 'model_weights.zip')))
display(FileLink(str(workspace / 'mlflow_results.zip')))